# GIK-IceChain — Quickstart

Minimum steps to run the full C1→C2→C3 pipeline on a small synthetic dataset.
**Prerequisites**: `pip install -e '.[dev]'` from the repo root.

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
import tempfile

WORK_DIR = Path(tempfile.mkdtemp(prefix='gik_quickstart_'))
print('Working directory:', WORK_DIR)

NLAT, NLON, NMEMBERS, NSTEPS = 8, 8, 10, 16
LAT   = np.linspace(0.0, 4.0, NLAT, dtype=np.float32)
LON   = np.linspace(35.0, 39.0, NLON, dtype=np.float32)
STEPS = np.arange(0, NSTEPS * 6, 6, dtype=np.int32)
WINDOWS_H      = [24, 72, 168]
RETURN_PERIODS = [5, 20]

## 1. Synthetic Forecast Dataset (C1 input)

In [ ]:
rng = np.random.default_rng(42)
tp  = np.cumsum(
    rng.exponential(0.003, (NMEMBERS, NSTEPS, NLAT, NLON)).astype(np.float32),
    axis=1,
)

forecast_ds = xr.Dataset({'tp': xr.DataArray(
    tp,
    dims=['member', 'step', 'latitude', 'longitude'],
    coords={'member': np.arange(NMEMBERS), 'step': STEPS, 'latitude': LAT, 'longitude': LON},
    attrs={'units': 'm'},
)})
print(forecast_ds)

## 2. Component 2: Rolling Accumulations & Exceedance Probabilities

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations

acc = compute_rolling_accumulations(forecast_ds, windows_h=WINDOWS_H)
for w in WINDOWS_H:
    key = f'tp_{w}h'
    print(f'{key}: shape={acc[key].shape}, max={float(acc[key].max()):.4f} m')

In [ ]:
from gik_icechain.exceedance.thresholds import AdaptiveGEVThresholds, ClimateMode, ENSOPhase, IODPhase, Season
from gik_icechain.exceedance.exceedance import compute_exceedance_probabilities, compute_ensemble_confidence


def _make_thresholds(
    lat: np.ndarray,
    lon: np.ndarray,
    windows_h: list,
    return_periods: list,
) -> AdaptiveGEVThresholds:
    thr = AdaptiveGEVThresholds()
    template = xr.DataArray(
        np.full((len(lat), len(lon)), 0.001, dtype=np.float32),
        dims=['latitude', 'longitude'],
        coords={'latitude': lat, 'longitude': lon},
    )
    for season in Season:
        for enso in ENSOPhase:
            for iod in IODPhase:
                mode = ClimateMode(season, enso, iod)
                thr._thresholds[mode.key] = {
                    w: {rp: template * (rp / 5.0) for rp in return_periods}
                    for w in windows_h
                }
    return thr


thr_inst = _make_thresholds(LAT, LON, WINDOWS_H, RETURN_PERIODS)
mode     = ClimateMode(Season.OND, ENSOPhase.NEUTRAL, IODPhase.NEUTRAL)

thr = thr_inst.get(24, 5, mode)
p   = compute_exceedance_probabilities(acc, xr.Dataset({'rp_5y': thr}), 24, 5, 'member')
assert float(p.min()) >= 0.0 and float(p.max()) <= 1.0
print(f'Exceedance prob (24h/5yr): min={float(p.min()):.3f}, max={float(p.max()):.3f} — OK')

conf = compute_ensemble_confidence(acc, window_h=24, member_dim='member')
unique_conf = set(int(v) for v in np.unique(conf.values))
assert unique_conf.issubset({0, 1, 2})
print(f'Ensemble confidence states: {sorted(unique_conf)} — OK')

## 3. Component 3: CRMA Risk Inference

In [ ]:
try:
    import pgmpy
    from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

    model = CRMAModel()
    model.build()

    evidence = CRMAEvidence(
        exceedance_prob_24h_5y=0.35,
        exceedance_prob_72h_5y=0.25,
        exceedance_prob_7d_5y=0.18,
        gpm_obs_24h=15.0,
        api_mm=55.0,
        spatial_coverage_fraction=0.45,
        consecutive_signal_days=2,
        sat_consecutive_days=1,
    )
    result = model.infer(evidence)
    print(f"Risk label : {result['risk_label']}")
    print(f"Green={result['p_green']:.3f}  Yellow={result['p_yellow']:.3f}  "
          f"Orange={result['p_orange']:.3f}  Red={result['p_red']:.3f}")
    total = result['p_green'] + result['p_yellow'] + result['p_orange'] + result['p_red']
    assert abs(total - 1.0) < 1e-4
    print('Probabilities sum to 1.0 — OK')
except ImportError:
    print('pgmpy not installed — skipping CRMA inference (install with: pip install pgmpy)')

## 4. Cleanup

All checks passed. See notebooks 01–03 for detailed per-component walkthroughs.

In [ ]:
import shutil
shutil.rmtree(WORK_DIR)
print('Cleaned up:', WORK_DIR)